# Nama: Syauqi Nabil Tasri

# NRP: 5054241040

<center><strong>Natural Language Processing</strong><br />
<strong><font color="blue">Semester Genap ITS</font></strong><br />
</center>

<strong>Outline pertemuan</strong><br />
<li> Word Embedding </li>

### Word Embedding

<p> Representasi dari teks ke dalam dense matriks. Term yang memiliki makna serupa berada pada jarak yang berdekatan pada ruang vektor </p>
<p> VSM merepresentasikan teks ke dalam sparse matriks </p>
<p> Sparse matriks memiliki ukuran yang besar dan memiliki banyak nilai 0 </p>
<p> Dense matriks ukurannya lebih kecil, dapat diinisialisasi, dan tidak memiliki nilai 0 </p>
<h2><img alt="" src="figures/3_word_embeddings.png" style="height: 296px ; width: 602px" /></h2>

#### Algoritma word embedding
<ul>
    <li> Skip-gram </li>
    <li> CBoW (Continuous Bag of Words)</li>
    <li> GloVe (Global Vector)</li>
    <li> FastText </li>
    <li> BERT (based on context) </li>
</ul>

#### Skip Gram dan CBoW
<h2><img alt="" src="figures/sg-cbow.png"/></h2>

<p> Dimana w(t) merupakan kata target dan w(t-n) - w(t+n) adalah konteks (kata yang berada disekitar kata target. n merupakan ukuran dari window size. Apabila n = 2 maka konteks didefinisikan sebagai 2 kata sebelum dan setelah kata target.</p>
<p> Contoh: "the big brown bear and the fox"
    <br>Apabila kata target adalah = "brown" dan n = 2
    <br>Maka:
    <br>w(t) = "brown"
    <br>w(t-2) = "the"
    <br>w(t-1) = "big"
    <br>w(t+1) = "bear"
    <br>w(t+2) = "and"
</p>
    

In [3]:
# download 3 categories from 20newgroup dataset
# save it as binary

import os
import pickle
from sklearn.datasets import fetch_20newsgroups

categories = ['sci.med', 'talk.politics.misc',  'rec.autos']
data = fetch_20newsgroups(subset='train', categories=categories,remove=('headers', 'footers', 'quotes'))

dst_name = "20newsgroup.pckl"
dst_path = os.path.join("data", dst_name)
with open(dst_path, 'wb') as fout:
    pickle.dump(data, fout)

In [4]:
# loada data pickle
with open(dst_path, 'rb') as fin:
    data = pickle.load(fin)

data = [doc for doc in data.data]

In [5]:
data[6]

'\nOk boys and girls,\n\n"What was the \'Ogadan War\'????"\n\nThe Money Raised in Band-Aid covered How Much of\nthe Cost of Which Soviet Client State to replace what\ncatagory of weapon system lost in the aforementioned war?\n\nWhy was the Joke: "We arm the World." Really Not that funny?\n\nGonzo Station is the designation for WHICH USN Op Area?\nand the primary threat targets in the Area Were:.....\n\nciao\ndrieux\n\n\n'

In [6]:
pip install nltk

  Using cached nltk-3.9.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached regex-2026.2.28-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.9.3-py3-none-any.whl (1.5 MB)
Using cached regex-2026.2.28-cp312-cp312-win_amd64.whl (277 kB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# preprocessing
import re
from nltk import sent_tokenize
from nltk import word_tokenize

def preprocess(doc):
    sents = sent_tokenize(doc)
    sents_tok = list() # tokenisasi kalimat
    for s in sents:
        s = s.strip().lower() # case folding dan menghilangkan new line
        s = s.replace("\n", " ") # menggantikan \n dengan spasi
        s = re.sub(r'[^a-zA-Z0-9 ]', ' ', s) # menghapus simbol
        s = re.sub(' +', ' ', s) # menghapus repetitive space

        sents_tok.append(s)

    return " ".join(sents_tok)

docs = list()
for d in data:
    docs.append(preprocess(d))
#docs

In [7]:
# Untuk membentuk word embedding digunakan library Gensim
# Format inputan dari Gensim adalah list of words untuk setiap kalimat
# Contoh pada dokumen:
#       Tiger is the biggest cat alive. The tigers species can be divided into 5 groups.
# Inputan yang dibutuhkan Gensim adalah:
#       ["tiger", "is", "the", "biggest", "cat", "alive"], ["the", "tigers", "species",
#        "can", "be", "devided", "into", "5", "groups"]
#
doc_gensim = [word_tokenize(d) for d in docs]
#print(doc_gensim)

In [11]:
# train skip-gram dengan library gensim
#
from gensim.models import Word2Vec

d = 300
model_sg = Word2Vec(doc_gensim, min_count=1, vector_size=d, window=5, workers=-1,)
print("DONE")

DONE


In [14]:
import os

model_path = os.path.join('model', 'model_sg.model')
# model dari word embedding dapat disimpan dengan fungsi di bawah ini
model_sg.save(model_path)

# untuk menggunakan model yang telah disimpan sebelumnya dapat dilakukan
# dengan memanfaatkan fungsi load, beberapa model membutuhkan waktu yang lama
# karena memiliki ukuran yang besar
model_sg_load = Word2Vec.load(model_path)

print('DONE')


DONE


#### Word2 Vec menggunakan matriks dense

<p>Penggunaan memori oleh Gensim</p>
<p>Jumlah kata x size x 12 bytes <br>
    Contoh: <br>
    Jika terdapat 100.000 kata unik dengan menggunakan dimensi embedding 200 <br>
    Maka memori yang dibutuhkan = 100.000 x 200 x 12 bytes = ~229MB
</p>

In [17]:
# Melihat vektor suatu kata
model_sg.wv['car']

array([-1.00789906e-03,  1.24398747e-03,  1.28387695e-03,  8.00735957e-04,
        2.50597997e-03,  1.16694486e-03,  1.93064054e-03,  1.89882435e-03,
       -3.04068369e-03,  2.93875928e-03, -5.12702856e-04, -1.75085943e-03,
       -2.74773710e-03,  9.87659674e-04, -1.30588806e-03,  3.28878569e-03,
        3.32316151e-03,  2.13594944e-03,  2.81202781e-04,  2.74087186e-04,
       -3.25250067e-03,  3.19459243e-03,  2.23949784e-03, -2.97298143e-03,
       -2.15306482e-03,  1.79569842e-03, -1.00471974e-04, -1.75808347e-03,
        1.12742418e-03, -3.11067095e-03,  2.13472056e-03,  8.86111637e-04,
        1.02987047e-03,  2.64722318e-03,  9.34786396e-04, -2.31564487e-03,
        5.64463553e-04, -8.58604093e-04, -1.86267891e-03,  1.79206172e-03,
        3.15546981e-06, -1.34923859e-04, -1.46604900e-03,  5.60994959e-04,
       -8.84941837e-04,  1.17924134e-03,  2.61398163e-05, -4.17157018e-04,
       -3.44450091e-04,  3.08117736e-03,  2.83078477e-03,  1.08563341e-03,
        1.72415341e-03, -

In [18]:
# Menghitung similarity vektor antar kata

model_sg.wv.similarity('car', 'car')

np.float32(1.0)

In [19]:
#  Menampilkan top-N similar words

model_sg.wv.similar_by_word('car', topn=10)

[('orders', 0.22770151495933533),
 ('860', 0.21312053501605988),
 ('flee', 0.21017464995384216),
 ('repairs', 0.2100219428539276),
 ('oxides', 0.20563443005084991),
 ('anyhow', 0.20279355347156525),
 ('period', 0.20236560702323914),
 ('rats', 0.19984197616577148),
 ('nother', 0.19766603410243988),
 ('que', 0.19389435648918152)]

In [20]:
# Mencari yang paling mirip dengan kata 'cars'
model_sg.wv.most_similar('cars')

[('nature', 0.22341984510421753),
 ('bipartisan', 0.2167140543460846),
 ('method', 0.21011313796043396),
 ('pucky', 0.20862150192260742),
 ('ceaseless', 0.20509134232997894),
 ('oppsed', 0.20361432433128357),
 ('myriad', 0.20207270979881287),
 ('veneer', 0.20017346739768982),
 ('parents', 0.19408173859119415),
 ('epa', 0.18759071826934814)]

<p><img alt="" src="figures/3_cosine.png" style="height:400px; width:683px" /></p>

In [21]:
# error jika kata tidak ada di training data

kata = 'entir'
try:
    print(model_sg.wv.most_similar(kata))
except:
    print('error! kata "',kata,'" tidak ada di training data')

error! kata " entir " tidak ada di training data


### Note: ###
<ul>
    <li> Pada word2vec apabila suatu kata tidak terdapat dalam list vocabulary maka akan error </li>
    <li> Kesalahan pengetikan (typo) bisa meninmbulkan error </li>
    <li> Untuk mangatasi hal ini dapat dilakukan dengan menggunakan algoritma lain, seperti FastText </li>

In [22]:
# "predict" vector for new data without re-training from the beginning
d1 = ['new','generation','nvidia','gpu','is','rtx']
d2 = ['deep','learning','computation','mostly', 'on', 'gpu']
d3 = ['the','rtx','gpu','capable','super','sampling','ssdl']
D = [d1,d2,d3]
model_sg.train(D, total_examples=len(D), epochs=model_sg.epochs)
print(model_sg.predict_output_word('gpu'))

[('tragics', np.float32(4.4517652e-05)), ('cruddy', np.float32(4.4517652e-05)), ('wibble', np.float32(4.4517652e-05)), ('wobble', np.float32(4.4517652e-05)), ('exchanging', np.float32(4.4517652e-05)), ('liscenced', np.float32(4.4517652e-05)), ('reprehensible', np.float32(4.4517652e-05)), ('interacts', np.float32(4.4517652e-05)), ('receptive', np.float32(4.4517652e-05)), ('vegan', np.float32(4.4517652e-05))]


<h3 id="Latihan:"><font color="blue">Latihan 1:</font></h3>

<ul>
    <li> Bangunlah word embedding dengan data 3 kategori lainnya dari data 20newsGroup</li>
    <li> Lakukan training menggunakan Skip-Gram dan CBoW </li>
    <li> Apakah vektor yang dihasilkan oleh suatu kata sama? ketika menggunakan Skip-Gram dan CBoW </li>
    <li> Apakah hasil top-n similar memperoleh vocabulary yang sama? </li>
    <li> Apakah jumlah dimensi berpengaruh pada hasil top-n most similar? </li>

In [23]:
import re
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

In [24]:
categories = [
    "sci.space",
    "rec.sport.baseball",
    "talk.politics.mideast"
]

news = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=42
)

print("Jumlah dokumen:", len(news.data))
print("Kategori:", news.target_names)
print("\nContoh dokumen pertama:\n")
print(news.data[0][:800])

Jumlah dokumen: 1754
Kategori: ['rec.sport.baseball', 'sci.space', 'talk.politics.mideast']

Contoh dokumen pertama:

Reply address: mark.prado@permanet.org

 > From: higgins@fnalf.fnal.gov (Bill Higgins-- Beam Jockey)
 >
 > In article <1993Apr19.230236.18227@aio.jsc.nasa.gov>,
 > > |> AW&ST  had a brief blurb on a Manned Lunar Exploration
 > confernce> |> May 7th  at Crystal City Virginia, under the
 > auspices of AIAA.
 >
 > Thanks for typing that in, Steven.
 >
 > I hope you decide to go, Pat.  The Net can use some eyes
 > and ears there...

I plan to go.  It's about 30 minutes away from my home.
I can report on some of it (from my perspective ...)
Anyone else on sci.space going to be there?  If so, send me
netmail.  Maybe we can plan to cross paths briefly...
I'll maintain a list of who's going.

mark.prado@permanet.org


In [25]:
def preprocess_text(text):
    # lowercase + tokenisasi sederhana
    tokens = simple_preprocess(text, deacc=True, min_len=2)
    return tokens

doc_gensim = [preprocess_text(doc) for doc in news.data]
doc_gensim = [doc for doc in doc_gensim if len(doc) > 0]

def train_word2vec(sentences, vector_size=100, sg=0, window=5, min_count=5, epochs=10, seed=42):
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        sg=sg,          # 0 = CBOW, 1 = Skip-Gram
        epochs=epochs,
        seed=seed
    )
    return model

dims = [50, 100, 300]

models = {}

for d in dims:
    models[("cbow", d)] = train_word2vec(doc_gensim, vector_size=d, sg=0)
    models[("skipgram", d)] = train_word2vec(doc_gensim, vector_size=d, sg=1)
    print(f"Selesai training CBOW dan Skip-Gram untuk dimensi {d}")

Selesai training CBOW dan Skip-Gram untuk dimensi 50
Selesai training CBOW dan Skip-Gram untuk dimensi 100
Selesai training CBOW dan Skip-Gram untuk dimensi 300


In [27]:
candidate_words = ["space", "game", "team", "year", "government", "people"]

test_word = None
for w in candidate_words:
    if all(w in models[(algo, d)].wv.key_to_index for algo in ["cbow", "skipgram"] for d in dims):
        test_word = w
        break

print("Kata uji:", test_word)

Kata uji: space


In [28]:
word = test_word
d = 100

vec_cbow = models[("cbow", d)].wv[word]
vec_sg   = models[("skipgram", d)].wv[word]

print("Kata:", word)
print("Shape CBOW:", vec_cbow.shape)
print("Shape Skip-Gram:", vec_sg.shape)

same_exact = np.array_equal(vec_cbow, vec_sg)
same_close = np.allclose(vec_cbow, vec_sg)

cos_sim = np.dot(vec_cbow, vec_sg) / (np.linalg.norm(vec_cbow) * np.linalg.norm(vec_sg))

print("Apakah persis sama?", same_exact)
print("Apakah hampir sama (allclose)?", same_close)
print("Cosine similarity antar-vektor:", cos_sim)

Kata: space
Shape CBOW: (100,)
Shape Skip-Gram: (100,)
Apakah persis sama? False
Apakah hampir sama (allclose)? False
Cosine similarity antar-vektor: 0.6434602


In [29]:
topn = 10
d = 100
word = test_word

cbow_top = models[("cbow", d)].wv.most_similar(word, topn=topn)
sg_top   = models[("skipgram", d)].wv.most_similar(word, topn=topn)

cbow_words = [w for w, s in cbow_top]
sg_words   = [w for w, s in sg_top]

print("Top-n CBOW:")
print(cbow_top)

print("\nTop-n Skip-Gram:")
print(sg_top)

common_words = sorted(set(cbow_words) & set(sg_words))
only_cbow = sorted(set(cbow_words) - set(sg_words))
only_sg   = sorted(set(sg_words) - set(cbow_words))

print("\nKata yang sama di top-n keduanya:", common_words)
print("Hanya muncul di CBOW:", only_cbow)
print("Hanya muncul di Skip-Gram:", only_sg)

Top-n CBOW:
[('nasa', 0.8720618486404419), ('shuttle', 0.8311762809753418), ('sci', 0.8144382834434509), ('funding', 0.7938910722732544), ('commercial', 0.7911568284034729), ('launch', 0.7792052030563354), ('program', 0.7718982696533203), ('project', 0.7691444754600525), ('astro', 0.7651699185371399), ('cost', 0.7584565877914429)]

Top-n Skip-Gram:
[('sci', 0.6696738600730896), ('astro', 0.5988903641700745), ('aviation', 0.5932284593582153), ('technical', 0.5846885442733765), ('aerospace', 0.5844944715499878), ('ssf', 0.578376829624176), ('commerce', 0.5705807209014893), ('monthly', 0.5695500373840332), ('station', 0.5640392303466797), ('shuttle', 0.5637466311454773)]

Kata yang sama di top-n keduanya: ['astro', 'sci', 'shuttle']
Hanya muncul di CBOW: ['commercial', 'cost', 'funding', 'launch', 'nasa', 'program', 'project']
Hanya muncul di Skip-Gram: ['aerospace', 'aviation', 'commerce', 'monthly', 'ssf', 'station', 'technical']


In [30]:
word = test_word
topn = 10

for algo in ["cbow", "skipgram"]:
    print(f"\n=== {algo.upper()} ===")
    results = {}
    for d in dims:
        sim_words = models[(algo, d)].wv.most_similar(word, topn=topn)
        results[d] = [w for w, s in sim_words]
        print(f"\nDimensi {d}:")
        print(sim_words)

    # overlap antardimensi
    base = set(results[dims[0]])
    for d in dims[1:]:
        overlap = base & set(results[d])
        print(f"\nOverlap top-{topn} antara dimensi {dims[0]} dan {d}: {sorted(overlap)}")
        print(f"Jumlah overlap: {len(overlap)}")


=== CBOW ===

Dimensi 50:
[('nasa', 0.858075737953186), ('shuttle', 0.8224645256996155), ('sci', 0.7896059155464172), ('launch', 0.7863525152206421), ('commercial', 0.7800682783126831), ('software', 0.7693095803260803), ('astro', 0.7660351395606995), ('funding', 0.7639027237892151), ('data', 0.7575382590293884), ('program', 0.7481066584587097)]

Dimensi 100:
[('nasa', 0.8720618486404419), ('shuttle', 0.8311762809753418), ('sci', 0.8144382834434509), ('funding', 0.7938910722732544), ('commercial', 0.7911568284034729), ('launch', 0.7792052030563354), ('program', 0.7718982696533203), ('project', 0.7691444754600525), ('astro', 0.7651699185371399), ('cost', 0.7584565877914429)]

Dimensi 300:
[('shuttle', 0.8756423592567444), ('nasa', 0.8616769909858704), ('sci', 0.8458545804023743), ('commercial', 0.8372641801834106), ('data', 0.8205538988113403), ('launch', 0.8142240643501282), ('funding', 0.8060521483421326), ('program', 0.8057113289833069), ('cost', 0.800362229347229), ('station', 0.798

In [31]:
word = test_word
topn = 10

summary_rows = []

for algo in ["cbow", "skipgram"]:
    for d in dims:
        sim_words = models[(algo, d)].wv.most_similar(word, topn=topn)
        words_only = [w for w, _ in sim_words]
        summary_rows.append({
            "model": algo,
            "dimensi": d,
            "kata_uji": word,
            "top_words": ", ".join(words_only)
        })

summary_df = pd.DataFrame(summary_rows)
summary_df

,model,dimensi,kata_uji,top_words
0,cbow,50,space,"nasa, shuttle, sci, launch, commercial, softwa..."
1,cbow,100,space,"nasa, shuttle, sci, funding, commercial, launc..."
2,cbow,300,space,"shuttle, nasa, sci, commercial, data, launch, ..."
3,skipgram,50,space,"sci, program, aerospace, astro, aviation, tech..."
4,skipgram,100,space,"sci, astro, aviation, technical, aerospace, ss..."
5,skipgram,300,space,"sci, astro, monthly, aerospace, publishes, exp..."


Berdasarkan eksperimen pada tiga kategori dataset 20 Newsgroups, embedding untuk kata yang sama, yaitu “space”, tidak identik antara model CBOW dan Skip-Gram. Hal ini terlihat dari nilai array_equal=False, allclose=False, dan cosine similarity sebesar 0.643, yang menunjukkan bahwa keduanya masih berhubungan tetapi tetap merepresentasikan kata dalam ruang vektor yang berbeda. Perbedaan ini wajar karena pada gensim.Word2Vec, sg=0 digunakan untuk CBOW dan sg=1 untuk Skip-Gram, sehingga kedua model belajar dengan objective training yang berbeda. Selain itu, hasil top-10 most similar juga tidak sama, karena pada dimensi 100 hanya ada tiga kata yang overlap, yaitu astro, sci, dan shuttle, sementara kata-kata lainnya berbeda di masing-masing model. Jumlah dimensi embedding juga terbukti memengaruhi hasil kemiripan kata, karena daftar top-10 similar pada dimensi 50, 100, dan 300 berubah walaupun masih memiliki sebagian overlap. Dengan demikian, dapat disimpulkan bahwa arsitektur Word2Vec dan jumlah dimensi embedding sama-sama berpengaruh terhadap representasi kata serta hasil top-n similar yang diperoleh.

## FastText (Facebook-2016)##
<ul>
    <li> Menggunakan Sub-words: app, ppl, ple - apple, sehingga dapat mengatasi permasalahan typo atau tidak adanya suatu kata dalam vocabularies </li>
    <li> Paper: https://arxiv.org/abs/1607.04606  </li>
    <li> Website: https://fasttext.cc/ </li>
    <li> Source: https://github.com/facebookresearch/fastText </li>
</ul>

In [33]:
from gensim.models import FastText

dim = 300 # Jumlah neurons = ukuran vektor = jumlah kolom
model_ft = FastText(doc_gensim, vector_size=dim, window=5, min_count=2, workers=-1)
'Done'

'Done'

In [34]:
#  Menampilkan top-N similar words

model_ft.wv.similar_by_word('car', topn=10)

[('care', 0.4145413041114807),
 ('cars', 0.3345319628715515),
 ('carew', 0.3314761817455292),
 ('carlos', 0.3219766616821289),
 ('carney', 0.3194786012172699),
 ('oscar', 0.3163032829761505),
 ('carbon', 0.30370697379112244),
 ('card', 0.30291464924812317),
 ('carving', 0.2988404333591461),
 ('carved', 0.29718315601348877)]

In [35]:
# Mencari yang paling mirip dengan kata 'cars'
model_ft.wv.most_similar('cars')

[('car', 0.33453187346458435),
 ('card', 0.3149397671222687),
 ('kars', 0.3005557060241699),
 ('pulsars', 0.29224133491516113),
 ('carl', 0.2877196669578552),
 ('care', 0.28634747862815857),
 ('wars', 0.2809792459011078),
 ('mars', 0.2674996554851532),
 ('carved', 0.26132163405418396),
 ('carry', 0.25524139404296875)]

In [36]:
# Melihat similarity antar kata

print(model_ft.wv.similarity('cars', 'car'))
print(model_sg.wv.similarity('cars', 'car'))

0.33453196
-0.09479238


In [37]:
# Tidak error jika kata tidak ada di training data

kata = 'beckmans'
try:
    print(model_ft.wv.most_similar(kata))
except:
    print('error! kata "',kata,'" tidak ada di training data')


# Tidak terjadi error saat kata tidak terdapat pada vocabulary

[('beck', 0.4223770499229431), ('jackman', 0.3106810748577118), ('humans', 0.2859816253185272), ('egyptians', 0.2526729702949524), ('ottomans', 0.2515616714954376), ('wickman', 0.23820087313652039), ('nervous', 0.2369116097688675), ('organs', 0.22717052698135376), ('bell', 0.22598206996917725), ('deck', 0.21907971799373627)]


<h3 id="Latihan:"><font color="blue">Latihan 2:</font></h3>

<ul>
	<li>Apakah kelebihan dan kekurangan WE secara umum?</li>
	<li>Apakah kira-kira aplikasi yang dapat memanfaatkan WE?</li>
	<li>Apakah bisa dijadikan representasi dokumen? Bagaimana caranya?</li>
	<li>Bergantung pada apa sajakah performa model WE?</li>
</ul>

* Preprocessing apa yang sebaiknya dilakukan pada model Word Embedding?
* Apakah Pos Tag bermanfaat disini? Jika iya bagaimana menggunakannya?


Jawaban:

1. **Kelebihan:** Word Embedding merepresentasikan kata sebagai vektor dense berdimensi rendah yang dapat menangkap kemiripan semantik dan sintaktik, sehingga kata-kata yang sering muncul dalam konteks mirip cenderung punya representasi yang berdekatan. Model seperti Word2Vec juga relatif efisien untuk dilatih pada korpus besar. **Kekurangan:** embedding klasik bersifat **static**, sehingga satu kata biasanya hanya punya satu vektor walaupun maknanya bisa berbeda tergantung konteks; selain itu, Word2Vec biasa juga tidak memodelkan subword secara eksplisit, sehingga kata langka atau OOV lebih sulit ditangani dibanding FastText.

2. WE dapat dimanfaatkan untuk banyak tugas NLP, misalnya **mengukur kemiripan kata/teks**, **rekomendasi**, **duplicate detection**, **clustering**, **text classification**, dan **sentiment analysis**. Dokumentasi spaCy secara eksplisit memberi contoh penggunaan similarity untuk rekomendasi dan penandaan duplikat, sedangkan paper Paragraph Vector menunjukkan bahwa representasi vektor juga efektif untuk tugas klasifikasi dokumen dan sentiment analysis.

3. **Bisa.** Cara paling sederhana adalah **merata-ratakan vektor token** dalam dokumen; di spaCy, `Doc.vector` memang secara default dihitung dari rata-rata vektor token. Cara yang lebih khusus adalah memakai **Doc2Vec / Paragraph Vector**, yaitu model yang memang dirancang untuk menghasilkan satu vektor dense per dokumen. Namun, rata-rata vektor punya kelemahan: ia tidak peka terhadap urutan kata dan kadang kurang mewakili makna frasa secara utuh.

4. Performa WE sangat bergantung pada **korpus latih** dan **hyperparameter**. Secara praktis, kualitas embedding dipengaruhi oleh ukuran dan kecocokan domain korpus, lalu parameter seperti `vector_size`, `window`, `min_count`, jumlah `epochs`, dan pilihan algoritma `sg=0` (CBOW) atau `sg=1` (Skip-Gram). FastText juga dipengaruhi oleh representasi subword, yang membantu untuk kata langka dan OOV.

5. Preprocessing yang umumnya baik adalah **tokenisasi yang konsisten**, **sentence splitting**, **normalisasi ringan** seperti lowercasing bila kapitalisasi tidak penting, serta **pembersihan noise** yang jelas tidak berguna seperti tag HTML atau karakter sangat kotor. Gensim juga menyediakan utilitas preprocessing seperti `lower_to_unicode`, `remove_stopwords`, dan `preprocess_documents`. Tetapi untuk WE, preprocessing jangan terlalu agresif, karena model belajar dari **konteks lokal dalam kalimat**; jadi menghapus terlalu banyak token konteks bisa mengurangi informasi semantik. Ini adalah inferensi dari fakta bahwa Word2Vec dilatih dari urutan token dalam kalimat dengan jendela konteks (`window`).

6. **Ya, bermanfaat**, terutama untuk **mengurangi ambiguitas makna kata**. POS tagging memberi label kelas kata seperti noun, verb, adjective, dan lain-lain. Salah satu cara pakainya adalah menggabungkan token dengan tag-nya, misalnya `book_NOUN` dan `book_VERB`, sehingga model belajar representasi yang berbeda untuk penggunaan yang berbeda. Dokumentasi spaCy juga menyinggung pendekatan seperti **sense2vec**, yang memang menggabungkan frasa dengan **POS tags** dan label entitas agar vektornya lebih spesifik.

Contoh sederhana:
```python
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp("They book a room and read a book.")
tokens_pos = [f"{tok.text.lower()}_{tok.pos_}" for tok in doc if tok.is_alpha]
print(tokens_pos)
```